In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("ingested_ts", "")

# Read each widget's value into a variable
source_system = dbutils.widgets.get("source_system")
table_name    = dbutils.widgets.get("table_name")
ingested_ts   = dbutils.widgets.get("ingested_ts")

print(source_system, table_name, ingested_ts)

In [0]:
df=spark.read.format("parquet") \
    .load(f"abfss://landing@storagecaresync.dfs.core.windows.net/{source_system}/{table_name}/{ingested_ts}")

df.display(10)

In [0]:
from pyspark.sql.functions import lit,current_timestamp, to_timestamp
from datetime import datetime
from pyspark.sql import functions as F

ts_format = "yyyy-MM-dd-HH-mm-ss"  

df = df.withColumn("ingestion_ts", F.to_timestamp(F.lit(ingested_ts), ts_format)) \
       .withColumn("bronze_loaded_ts", current_timestamp())

In [0]:
df.write.format("delta").mode("append").saveAsTable(f"ws_caresync.bronze.{table_name}")

In [0]:
%sql
select count(*),ingestion_ts from ws_caresync.bronze.hospitals
group by ingestion_ts;